### [1] Functions

In [ ]:
import torch
torch.set_default_dtype(torch.float64)
for i in range(torch.cuda.device_count()):
    print(f'Device: {torch.cuda.get_device_properties(i).name}')
if torch.cuda.is_available():
    i = 0
    torch.cuda.set_device(i)
    device = f'cuda:{i}'
    torch.set_default_device(f'cuda:{i}')
    print(f"Cuda is available. Setting default device to: {torch.cuda.get_device_properties(i).name}")
else:
    print('Cuda is not available. Setting default device to: CPU')
    device = 'cpu'

import kan
import matplotlib.pyplot as plt
import numpy as np
import scipy
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error, max_error

Device: NVIDIA GeForce RTX 4060 Laptop GPU
Cuda is available. Setting default device to: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
# --- Create dataset
f = lambda x: x
dataset = kan.create_dataset(f, n_var=2, device=device,
                             train_num=1000, test_num=1000,
                             normalize_input=True, normalize_label=False)

ft = dataset['train_input']

In [ ]:
# --- Model Wrapper
class KanModel:
# Initialize model
    def __init__(
                 self,
                 dataset,
                 hidden_layers,
                 device,
                 **kwargs
                 ):
        n_inputs = dataset['train_input'].shape[1]
        n_outpts = dataset['train_label'].shape[1]
        width = [n_inputs] + hidden_layers + [n_outpts]

        self.model = kan.KAN(width=width, device=device, **kwargs)
        self.dataset = dataset
        self.history = {"train":[],"tests":[]}
        return
# Train model
    def fit(
            self,
            dataset = None,
            **kwargs
           ):
        results = self.model.fit(dataset, **kwargs)
        self.history['train'] += results['train_loss']
        self.history['tests'] += results['test_loss']
        return results
# Prune model
    def prune(
              self,
              node_th = None,
              edge_th = None,
              input_th = None,
              **kwargs
             ):
        if node_th:
            self.model = self.model.prune_node(node_th, **kwargs)
        if edge_th:
            self.model.prune_edge(edge_th)
        if input_th:
            self.model = self.model.prune_input(input_th)
        return
# Refine model
    def refine(
               self,
               num = 0,
               factor = 1
              ):
        new_grid = num if num else self.model.grid * factor 
        self.model = self.model.refine(new_grid)
        return
# Plot model training
    def plot_training(
                      self
                     ):
        fig, axs = plt.subplots(1,1)
        axs.plot([i for i in range(len(self.history["train"]))], self.history["train"])
        axs.plot([i for i in range(len(self.history["tests"]))], self.history["tests"])
        axs.set_yscale('log'); axs.set_ylabel('Loss');axs.set_xlabel("Steps"); axs.grid(visible=True, which='both')
        fig.tight_layout()
        return fig
# Plot model performance
    def plot_performance(
                         self,
                         dataset = None
                        ):
        
        Y_TRUE = dataset['test_label'] if dataset else self.dataset['test_label'].cpu().to_numpy()
        Y_PREDICTED = self.model(dataset['test_input']) if dataset else self.model(self.dataset['test_input']).detach().cpu().to_numpy()

        fig, axs = plt.subplots(1,2, figsize=(12,4))

        ax=axs[0]
        ax.scatter(Y_TRUE, Y_PREDICTED, alpha=0.25, label=f"Model")
        ax.plot(np.linspace(np.min(Y_TRUE), np.max(Y_TRUE),100),np.linspace(np.min(Y_TRUE), np.max(Y_TRUE),100), color='black',ls='--', label="True")
        ax.set_xlabel("f(x)"); ax.set_ylabel("KAN(x)"); ax.legend(); ax.grid(visible=True, color='gainsboro')

        ax=axs[1]
        residuals = Y_PREDICTED[:,0] - Y_TRUE
        pdf_fit = scipy.stats.gennorm.fit(residuals)
        pdf_pdf = scipy.stats.gennorm.pdf(np.linspace(-0.3,0.3,100), *pdf_fit)
        ax.hist(residuals, bins=100, color='cornflowerblue', density=True)
        ax.plot(np.linspace(-0.3,0.3,100), pdf_pdf, label=f'Fit: {[round(i,3) for i in pdf_fit]}')
        ax.set_xlabel("Residual"); ax.set_ylabel("Relative Frequency"); ax.grid(visible=True, color='gainsboro', label='Raw Data'); ax.legend()

        fig.tight_layout()
        return
# Evaluate model performance
    def evaluate_performance(
                            self,
                            dataset = None
                            ):
        Y_TRUE = dataset['test_label'] if dataset else self.dataset['test_label'].cpu().to_numpy()
        Y_PREDICTED = self.model(dataset['test_input']) if dataset else self.model(self.dataset['test_input']).detach().cpu().to_numpy()
        r2 = r2_score(Y_TRUE, Y_PREDICTED)
        mae = mean_absolute_error(Y_TRUE, Y_PREDICTED)
        mse = mean_squared_error(Y_TRUE, Y_PREDICTED)
        rmse = root_mean_squared_error(Y_TRUE, Y_PREDICTED)
        maxerr = max_error(Y_TRUE, Y_PREDICTED)
        print(f"R2: {r2}\tMAE: {mae}\tMSE: {mse}\tRMSE: {rmse}\tMaxAE: {maxerr}")
        return {'R2':r2, 'MAE':mae, 'MSE':mse, 'RMSE':rmse, 'MaxAE':maxerr}
# Plot model
    def plot(
             self,
             **kwargs
            ):
        self.model.plot(**kwargs)
        return
# Fix and obtain symbolic functions
    def symbolize(
                  self,
                  library = None,
                  **kwargs
                  ):
        self.model.auto_symbolic(library)
        return

In [ ]:
mykan = KanModel(dataset, [1,[2,2]], 'cuda:0')
mykan.fit(dataset)

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 3.21e-01 | test_loss: 3.19e-01 | reg: 2.18e+01 | : 100%|█| 100/100 [00:32<00:00,  3.09

saving model version 0.1


{'train_loss': [array(0.369302),
  array(0.35439006),
  array(0.34673719),
  array(0.34172027),
  array(0.33823588),
  array(0.34889025),
  array(0.342328),
  array(0.33919946),
  array(0.33701438),
  array(0.33572533),
  array(0.33850585),
  array(0.33693891),
  array(0.33564779),
  array(0.33454184),
  array(0.33327182),
  array(0.33341475),
  array(0.33145331),
  array(0.33103786),
  array(0.33064237),
  array(0.33017248),
  array(0.33090487),
  array(0.33048756),
  array(0.32999184),
  array(0.32895912),
  array(0.3276994),
  array(0.3286816),
  array(0.32677718),
  array(0.32622248),
  array(0.3259707),
  array(0.32575836),
  array(0.32683188),
  array(0.32658663),
  array(0.32621028),
  array(0.32598614),
  array(0.32545725),
  array(0.32615709),
  array(0.32607906),
  array(0.3259367),
  array(0.32581706),
  array(0.32561528),
  array(0.32573264),
  array(0.325516),
  array(0.32540657),
  array(0.32529702),
  array(0.32510972),
  array(0.32557077),
  array(0.32538809),
  array(0